# Boosting & Hyperparameter Tuning
This notebook compares a decision-tree baseline with AdaBoost, Gradient Boosting, and HistGradientBoosting, then tunes Gradient Boosting using cross-validation. Run cells in order.

## 1. Imports and dataset
The built-in Breast Cancer Wisconsin dataset avoids external file dependencies. The test set is held out until final evaluation.

In [ ]:
from time import perf_counter
import pandas as pd
from sklearn.datasets import load_breast_cancer
from sklearn.ensemble import AdaBoostClassifier, GradientBoostingClassifier, HistGradientBoostingClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.tree import DecisionTreeClassifier

data = load_breast_cancer()
X = pd.DataFrame(data.data, columns=data.feature_names)
y = pd.Series(data.target, name='target')
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print('Train shape:', X_train.shape)
print('Test shape:', X_test.shape)
print('Class proportions:', y_train.value_counts(normalize=True).round(3).to_dict())

## 2. Bagging vs boosting — concept check
**Bagging** trains models independently on resampled data and combines their predictions. **Boosting** builds learners sequentially so later learners focus on patterns the current ensemble still handles poorly. Random Forest is a common bagging example; AdaBoost and Gradient Boosting are boosting examples.

In [ ]:
models = {
    'Decision Tree': DecisionTreeClassifier(max_depth=4, random_state=42),
    'AdaBoost': AdaBoostClassifier(n_estimators=100, learning_rate=0.5, random_state=42),
    'Gradient Boosting': GradientBoostingClassifier(n_estimators=100, learning_rate=0.05, max_depth=2, random_state=42),
    'HistGradientBoosting': HistGradientBoostingClassifier(max_iter=100, learning_rate=0.1, max_leaf_nodes=15, random_state=42),
}

rows = []
fitted_models = {}
for name, model in models.items():
    start = perf_counter()
    model.fit(X_train, y_train)
    fit_seconds = perf_counter() - start
    fitted_models[name] = model
    pred = model.predict(X_test)
    score = model.predict_proba(X_test)[:, 1] if hasattr(model, 'predict_proba') else model.decision_function(X_test)
    rows.append({
        'Model': name,
        'Accuracy': accuracy_score(y_test, pred),
        'Precision': precision_score(y_test, pred, zero_division=0),
        'Recall': recall_score(y_test, pred, zero_division=0),
        'F1': f1_score(y_test, pred, zero_division=0),
        'ROC-AUC': roc_auc_score(y_test, score),
        'Fit seconds': fit_seconds,
    })

results = pd.DataFrame(rows).sort_values('F1', ascending=False)
results.round(4)

## 3. Hyperparameter tuning with RandomizedSearchCV
Search only within the training data. Cross-validation estimates training-set generalization; the held-out test set is not used to choose parameters.

In [ ]:
param_distributions = {
    'n_estimators': [50, 100, 150, 200],
    'learning_rate': [0.01, 0.03, 0.05, 0.1],
    'max_depth': [1, 2, 3],
    'subsample': [0.7, 0.85, 1.0],
}
search = RandomizedSearchCV(
    GradientBoostingClassifier(random_state=42),
    param_distributions=param_distributions,
    n_iter=15,
    scoring='f1',
    cv=5,
    random_state=42,
    n_jobs=-1,
)
search.fit(X_train, y_train)
print('Best parameters:', search.best_params_)
print('Best mean CV F1:', round(search.best_score_, 4))

## 4. Final test evaluation
Evaluate the selected model once on the held-out test split. A strong CV score does not guarantee an identical test score.

In [ ]:
best_model = search.best_estimator_
test_pred = best_model.predict(X_test)
test_score = best_model.predict_proba(X_test)[:, 1]
print('Test accuracy:', round(accuracy_score(y_test, test_pred), 4))
print('Test precision:', round(precision_score(y_test, test_pred, zero_division=0), 4))
print('Test recall:', round(recall_score(y_test, test_pred, zero_division=0), 4))
print('Test F1:', round(f1_score(y_test, test_pred, zero_division=0), 4))
print('Test ROC-AUC:', round(roc_auc_score(y_test, test_score), 4))

## 5. Optional extensions
- Try XGBoost (`pip install xgboost`) or LightGBM (`pip install lightgbm`) and add the model to the comparison table.
- Try Optuna (`pip install optuna`) to optimize a validation metric and log trial results.
- Investigate early stopping where the chosen estimator and API support it.

## Reflection
1. Explain bagging versus boosting.
2. Which model achieved the highest test F1 in your run?
3. Did tuning improve cross-validation performance, and did that improvement carry over to the test set?
4. Why must the test set not be used to select hyperparameters?